In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from students.kirienko.lesson2 import Exercise

In [ ]:
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ

iris = load_iris()
X = iris.data.astype(float)
y = iris.target.astype(float)
y_binary = (y == 0).astype(int)

# Разделение данных
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y_binary, test_size=0.4, random_state=42, stratify=y_binary
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.5, random_state=42, stratify=y_train_val
)

# Нормализация
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train_norm = (X_train - mean) / std
X_val_norm = (X_val - mean) / std
X_test_norm = (X_test - mean) / std

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
#  ГИПЕРПАРАМЕТРЫ ДЛЯ ПОИСКА
lr_values = [
    0.001,
    0.003,
    0.005,
    0.007,
    0.009,
    0.01,
    0.03,
    0.05,
    0.07,
    0.09,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
    1.0,
]
batch_sizes = [1, 2, 4, 8, 16, 32, 64]
seeds = [42, 222, 456, 666]

print(f"Learning rates: {len(lr_values)} значений")
print(f"Batch sizes: {batch_sizes}")
print(f"Seeds: {seeds}")
print(f"Всего комбинаций: {len(lr_values) * len(batch_sizes) * len(seeds)}")

In [ ]:
def fit_with_batches(model, X, y, lr, n_epochs, batch_size):
    n_samples = X.shape[0]
    for _ in range(n_epochs):
        #  БЕРЕМ данные последовательно
        for start in range(0, n_samples - batch_size + 1, batch_size):
            end = start + batch_size
            X_batch = X[start:end]
            y_batch = y[start:end]
            Exercise.fit(model, X_batch, y_batch, lr, 1)

In [ ]:
#  ПОИСК ГИПЕРПАРАМЕТРОВ
print("ПОИСК ЛУЧШИХ ГИПЕРПАРАМЕТРОВ ")

results = []
total = len(lr_values) * len(batch_sizes) * len(seeds)
current = 0

for lr in lr_values:
    for batch_size in batch_sizes:
        for seed in seeds:
            current += 1
            if current % 20 == 0:
                print(f"Прогресс: {current}/{total}")

            model = Exercise.create_logistic_model(num_features=4, rng=np.random.default_rng(seed))

            fit_with_batches(model, X_train_norm, y_train, lr, 25, batch_size)

            #  2 МЕТРИКИ
            val_accuracy = model.metric(X_val_norm, y_val, metric="accuracy")
            val_auroc = model.metric(X_val_norm, y_val, metric="AUROC")

            results.append(
                {"lr": lr, "batch_size": batch_size, "seed": seed, "val_accuracy": val_accuracy, "val_auroc": val_auroc}
            )

df_results = pd.DataFrame(results)

In [ ]:
#  АНАЛИЗ РЕЗУЛЬТАТОВ

print("ТОП-10 ЛУЧШИХ КОМБИНАЦИЙ ")
top10 = df_results.sort_values("val_auroc", ascending=False).head(10)
print(top10.to_string(index=False))

best = df_results.loc[df_results["val_auroc"].idxmax()]
print("ЛУЧШАЯ КОМБИНАЦИЯ:")
print(f"  Learning rate: {best['lr']}")
print(f"  Batch size:    {int(best['batch_size'])}")
print(f"  Seed:          {int(best['seed'])}")
print(f"  Accuracy:      {best['val_accuracy']:.4f}")
print(f"  AUROC:         {best['val_auroc']:.4f}")


print("СТАТИСТИКА ПО LEARNING RATES:")
lr_stats = (
    df_results.groupby("lr")["val_auroc"].agg(["mean", "max", "count"]).round(4).sort_values("mean", ascending=False)
)
print(lr_stats)

print("СТАТИСТИКА ПО BATCH SIZES:")
batch_stats = (
    df_results.groupby("batch_size")["val_auroc"]
    .agg(["mean", "max", "count"])
    .round(4)
    .sort_values("mean", ascending=False)
)
print(batch_stats)

In [ ]:
# ACCURACY
print("ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ")

final_model = Exercise.create_logistic_model(num_features=4, rng=np.random.default_rng(int(best["seed"])))

fit_with_batches(final_model, X_train_norm, y_train, best["lr"], 25, int(best["batch_size"]))

train_acc = final_model.metric(X_train_norm, y_train, metric="accuracy")
test_acc = final_model.metric(X_test_norm, y_test, metric="accuracy")

print("\nРезультаты финальной модели:")
print(f"  Accuracy на train: {train_acc:.4f}")
print(f"  Accuracy на test:  {test_acc:.4f}")

In [ ]:
# ПРОВЕРКА AUROC
print("ПРОВЕРКА AUROC ДЛЯ ЛУЧШЕЙ МОДЕЛИ")

train_auroc = final_model.metric(X_train_norm, y_train, metric="AUROC")
test_auroc = final_model.metric(X_test_norm, y_test, metric="AUROC")

print(f"  AUROC на train: {train_auroc:.4f}")
print(f"  AUROC на test:  {test_auroc:.4f}")

In [ ]:
batch_stats = df_results.groupby("batch_size")["val_accuracy"].agg(["mean", "max", "count"]).round(4)
batch_stats